ai-01

In [ ]:
import sys, json, types
lrn_llm = types.ModuleType("lrn_llm")
try:
    from pyodide.http import pyfetch as _pyfetch
    _IN_PYODIDE = True
except ImportError:
    import urllib.request as _urlreq
    _IN_PYODIDE = False
lrn_llm.API_BASE = "/api/llm"  # same-origin proxy; server injects the gateway key
lrn_llm.DEFAULT_MODEL = "azure/gpt-5.4-mini"
lrn_llm.API_KEY = ""  # optional; set in Step 0a

async def _lrn_call(messages, *, system=None, max_tokens=400, model=None):
    if system is not None:
        messages = [{"role": "system", "content": system}] + list(messages)
    payload = {"model": model or lrn_llm.DEFAULT_MODEL, "messages": messages,
               "max_completion_tokens": max_tokens}
    headers = {"content-type": "application/json"}
    _key = lrn_llm.API_KEY
    if _key:
        headers["Authorization"] = "Bearer " + _key
    url = lrn_llm.API_BASE.rstrip("/") + "/chat/completions"
    body = json.dumps(payload)
    if _IN_PYODIDE:
        r = await _pyfetch(url, method="POST", headers=headers, body=body)
        data = await r.json()
    else:
        req = _urlreq.Request(url, method="POST", headers=headers, data=body.encode("utf-8"))
        with _urlreq.urlopen(req, timeout=60) as r:
            data = json.loads(r.read())
    if "error" in data:
        raise RuntimeError("LLM error: " + str(data["error"]))
    return data

def _lrn_text(r):
    ch = (r or {}).get("choices") or []
    return (ch[0].get("message", {}) or {}).get("content", "") if ch else ""

async def _lrn_ping():
    r = await _lrn_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=5)
    return {"ok": _lrn_text(r).strip().upper().startswith("OK"), "model": r.get("model")}

lrn_llm.call = _lrn_call
lrn_llm.text = _lrn_text
lrn_llm.ping = _lrn_ping
print("✅ notebook ready · endpoint:", lrn_llm.API_BASE)

## Step 0a — Endpoint & Key

Set your API key if you have one (optional on the LHIND network). The default model is GPT-4o via Azure.

In [ ]:
# Optional: set your API key here if needed
lrn_llm.API_KEY = ""

print("Endpoint:", lrn_llm.API_BASE)
print("Model:", lrn_llm.DEFAULT_MODEL)
print("Authentication:", "API key set" if lrn_llm.API_KEY else "using gateway default")

## Step 1 — Reachability

Ping the LLM endpoint to confirm the connection works.

In [ ]:
r = await lrn_llm.ping()
print("✅ LLM reachable" if r["ok"] else "❌ LLM unreachable")
print(f"Model: {r['model']}")

## Step 2 — The Coding Task: FizzBuzz with a Bug

Imagine you inherit a codebase with a known bug. Your job: use two different prompt variants to ask an LLM to fix it. Compare the outputs.

Here is the buggy code (an off-by-one error in the range):

In [ ]:
buggy_code = '''def fizzbuzz(n):
    """Return FizzBuzz sequence for numbers 1 to n."""
    result = []
    for i in range(1, n):  # BUG: should be range(1, n+1)
        if i % 15 == 0:
            result.append("FizzBuzz")
        elif i % 3 == 0:
            result.append("Fizz")
        elif i % 5 == 0:
            result.append("Buzz")
        else:
            result.append(str(i))
    return result
'''

print("Buggy code:")
print(buggy_code)
print("\nBug: The range(1, n) excludes the last number n")
print("Example: fizzbuzz(5) returns ['1', '2', 'Fizz', '4'] instead of ['1', '2', 'Fizz', '4', 'Buzz']")

## Step 3 — Prompt Variant 1: Vague Instruction

This variant is generic and under-specified. It lacks role context, explicit constraints, and clear output format.

In [ ]:
variant_1_system = "You are a helpful AI assistant."

variant_1_user = f"""Fix the bug in this Python function:

{buggy_code}

Tell me what the bug is and provide the corrected code."""

print("=== VARIANT 1: VAGUE ===")
print(f"\nSystem: {variant_1_system}")
print(f"\nUser prompt:\n{variant_1_user}")

Send Variant 1 to the LLM and capture the response.

In [ ]:
resp_1 = await lrn_llm.call(
    [{"role": "user", "content": variant_1_user}],
    system=variant_1_system,
    max_tokens=400
)
output_1 = lrn_llm.text(resp_1)

print("LLM Response (Variant 1):")
print("-" * 60)
print(output_1)
print("-" * 60)

## Step 4 — Prompt Variant 2: Engineered Instruction

This variant applies prompt engineering patterns: specific role, explicit constraints, structured output format, and examples of what we want.

In [ ]:
variant_2_system = """You are a senior Python code reviewer. Your task is to identify bugs and provide corrected code that is clear, efficient, and follows PEP 8 conventions.

When fixing code:
- Identify the ROOT CAUSE of the bug, not just the symptom
- Provide the corrected code with NO surrounding explanation
- Verify your fix against the expected behavior
- Output ONLY the corrected function, no other text"""

variant_2_user = f"""Fix the bug in this FizzBuzz function. The function should return all numbers from 1 to n (inclusive).

Buggy code:
{buggy_code}

Expected behavior for fizzbuzz(5): ['1', '2', 'Fizz', '4', 'Buzz']

Provide ONLY the corrected function, starting with 'def fizzbuzz'."""

print("=== VARIANT 2: ENGINEERED ===")
print(f"\nSystem: {variant_2_system[:150]}...")
print(f"\nUser prompt:\n{variant_2_user}")

Send Variant 2 to the LLM and capture the response.

In [ ]:
resp_2 = await lrn_llm.call(
    [{"role": "user", "content": variant_2_user}],
    system=variant_2_system,
    max_tokens=400
)
output_2 = lrn_llm.text(resp_2)

print("LLM Response (Variant 2):")
print("-" * 60)
print(output_2)
print("-" * 60)

## Step 5 — Evaluate & Compare

Score both outputs on clarity, correctness, and format compliance. The engineered prompt should steer toward a cleaner, more focused response.

In [ ]:
def eval_response(text, variant_name):
    """Evaluate a response for correctness and clarity."""
    criteria = {}
    
    # Check if response contains 'range(1, n+1)' (the fix)
    criteria["has_fix"] = "range(1, n+1)" in text or "range(1,n+1)" in text
    
    # Check if it's mostly code (engineered variant should be concise)
    code_lines = len([l for l in text.split('\n') if l.strip().startswith('def') or l.strip().startswith('for') or l.strip().startswith('if')])
    criteria["has_code"] = code_lines > 0
    
    # Check verbosity (engineered variant should be shorter)
    word_count = len(text.split())
    criteria["word_count"] = word_count
    criteria["concise"] = word_count < 100  # arbitrary but reasonable threshold
    
    # Check for explanation text (engineered variant should have minimal explanation)
    has_explanation = any(phrase in text.lower() for phrase in 
        ["the bug is", "the issue is", "the problem is", "explanation:"])
    criteria["minimal_explanation"] = not has_explanation
    
    print(f"\n{variant_name}:")
    print(f"  ✓ Contains fix (range(1, n+1)): {criteria['has_fix']}")
    print(f"  ✓ Contains code: {criteria['has_code']}")
    print(f"  ✓ Word count: {criteria['word_count']} words")
    print(f"  ✓ Concise (< 100 words): {criteria['concise']}")
    print(f"  ✓ Minimal explanation: {criteria['minimal_explanation']}")
    
    # Composite score: how many criteria passed?
    passed = sum(1 for v in criteria.values() if isinstance(v, bool) and v)
    total_bool = sum(1 for v in criteria.values() if isinstance(v, bool))
    score = passed / total_bool if total_bool > 0 else 0
    print(f"  SCORE: {score:.2%} ({passed}/{total_bool} criteria)")
    
    return criteria, score

print("=" * 60)
print("EVALUATION")
print("=" * 60)

criteria_1, score_1 = eval_response(output_1, "Variant 1 (Vague)")
criteria_2, score_2 = eval_response(output_2, "Variant 2 (Engineered)")

print("\n" + "=" * 60)
print(f"WINNER: Variant {'2 (Engineered)' if score_2 > score_1 else '1 (Vague)'} with {max(score_1, score_2):.2%}")
print("=" * 60)

## Step 6 — Key Insights

The difference between Variant 1 and Variant 2 demonstrates the core principle of prompt engineering:

**Vague prompts (Variant 1)** activate a broad distribution of responses. The model may explain the bug, show before/after, add commentary, or use different formatting. All are "correct" but inconsistent.

**Engineered prompts (Variant 2)** constrain the output space by specifying:
- A specific role ("senior code reviewer")
- Explicit format instructions ("ONLY the corrected function")
- Expected behavior (the example output)
- What NOT to do ("no surrounding explanation")

The result: the model is more likely to return exactly what you need — clean, focused code — with fewer tokens and faster evaluation.

## Try It Yourself

Design your own prompt variants. The code below has a different bug (off-by-one in string slicing). Write two prompts:
1. A vague one (minimal context)
2. An engineered one (specific role, constraints, format)

Compare the outputs using the same evaluation framework.

In [ ]:
# NEW BUG: off-by-one in string slicing
test_code = '''def reverse_string(s):
    """Reverse a string."""
    return s[:len(s)-1]  # BUG: excludes last character when reversing
'''

print("Test code with a bug:")
print(test_code)
print("\nWhat is the bug? How would you fix it?")
print("\nWrite your two prompt variants below (replace the TODOs):")
print()

# TODO: Write your vague variant here
your_vague_system = "TODO: system message for vague variant"
your_vague_user = f"""TODO: user prompt for vague variant

{test_code}"""

# TODO: Write your engineered variant here
your_engineered_system = "TODO: system message for engineered variant"
your_engineered_user = f"""TODO: user prompt for engineered variant

{test_code}"""

print("After filling in the TODOs, uncomment the code below to test:")
print()
print("# resp_vague = await lrn_llm.call(")
print("#     [{\"role\": \"user\", \"content\": your_vague_user}],")
print("#     system=your_vague_system,")
print("#     max_tokens=400")
print("# )")
print("# output_vague = lrn_llm.text(resp_vague)")
print("# print(\"Vague output:\", output_vague)")
print()
print("# resp_eng = await lrn_llm.call(")
print("#     [{\"role\": \"user\", \"content\": your_engineered_user}],")
print("#     system=your_engineered_system,")
print("#     max_tokens=400")
print("# )")
print("# output_eng = lrn_llm.text(resp_eng)")
print("# print(\"Engineered output:\", output_eng)")